# 💬 Worker LIPSYNC — VideoClip Creator

**ANTES DE EJECUTAR (solo la 1ª vez):** menú `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

**Después:** pulsa **`Ctrl+F9`** (o `Entorno de ejecución → Ejecutar todas`) y deja esta pestaña abierta.

Cuando veas `🟢 WORKER ACTIVO` al final, este worker ya se habrá **auto-registrado** en tu orquestador: vuelve a la página Lanzador y verás el ✅.

> ⏱ La primera ejecución tarda 5-10 min (descarga del modelo). Si Colab se desconecta, vuelve a pulsar `Ctrl+F9` (el worker se re-registrará solo con su nueva URL).

In [ ]:
# CELDA 1 · Instalar Wav2Lip (5-10 min la 1ª vez)
!git clone -q https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
!pip install -q gdown "librosa==0.8.1" "numba==0.56.4" "opencv-python-headless==4.8.1.78" \
    fastapi "uvicorn[standard]" nest-asyncio
%cd /content/Wav2Lip
!gdown -q --fuzzy 'https://drive.google.com/file/d/15hwP0d7fyGkjb6IVmXqYPJB0TdkEecJ7/view' \
    -O /content/wav2lip_gan.pth || echo '⚠️ si falla, descarga wav2lip_gan.pth a mano'
!wget -q https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth \
    -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /content/cloudflared && chmod +x /content/cloudflared
import os
print('✅ instalación lista', '- checkpoint OK' if os.path.getsize('/content/wav2lip_gan.pth') > 1_000_000 else '- ⚠️ checkpoint pendiente')

In [ ]:
# CELDA 2 · Servidor del worker (endpoint /lipsync)
import base64, subprocess, threading, time, uuid
import nest_asyncio, uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import Response

TOKEN = 'b794a2e6f1c3d5b8a0e4f7c2d9b1a3e6f8d0c5b4a2e7f9d1c3b6a8e0f4d7dbb0'   # inyectado por generar_notebooks.py
app = FastAPI()

@app.get('/ping')
def ping():
    return {'ok': True, 'rol': 'lipsync'}

@app.post('/lipsync')
def lipsync(body: dict, x_token: str = Header(default='')):
    if x_token != TOKEN:
        raise HTTPException(401, 'token inválido')
    uid = uuid.uuid4().hex[:8]
    v, a, o = f'/content/{uid}.mp4', f'/content/{uid}.wav', f'/content/{uid}_out.mp4'
    open(v, 'wb').write(base64.b64decode(body['video_b64']))
    open(a, 'wb').write(base64.b64decode(body['audio_b64']))
    subprocess.run(['python', '/content/Wav2Lip/inference.py',
                    '--checkpoint_path', '/content/wav2lip_gan.pth',
                    '--face', v, '--audio', a, '--outfile', o,
                    '--pads', '0', '15', '0', '0'],
                   check=True, capture_output=True)
    return Response(content=open(o, 'rb').read(), media_type='video/mp4')

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8080), daemon=True).start()
time.sleep(5)
print('✅ servidor lipsync corriendo')

In [ ]:
# CELDA 3 · Túnel público + AUTO-REGISTRO en el orquestador
import subprocess, re, time, requests

proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8080'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(90):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', proc.stdout.readline())
    if m:
        url = m.group(0)
        break
assert url, 'No se pudo crear el túnel: reintenta esta celda'
print('🌐 URL pública del worker:', url)

API = 'https://api.ai-producer-2qd.pages.dev'
for intento in range(30):
    try:
        r = requests.post(API + '/api/workers/registrar',
                          json={'rol': 'lipsync', 'url': url},
                          headers={'X-Token': TOKEN}, timeout=20)
        print('✅ Auto-registrado en el orquestador:', r.json())
        break
    except Exception as e:
        print(f'  reintentando registro ({intento+1}/30)...', e)
        time.sleep(5)
else:
    print('❌ No se pudo registrar: revisa PUBLIC_API_URL y WORKER_TOKEN')

print()
print('🟢 WORKER ACTIVO. Vuelve al Lanzador: verás el check verde ✅')
print('   NO cierres esta pestaña mientras generas videoclips.')
while True:
    time.sleep(60)
